<a href="https://colab.research.google.com/github/sidhu2690/MARL/blob/main/Merkle_Tree_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
import itertools
import random
import time

MAX_LEAF = 4
MAX_DEPTH = 256


def h(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()


def entry_hash(entry_id, xy):
    return h(f"{entry_id}:{xy[0]:.6f}:{xy[1]:.6f}".encode())

def bit(hash_bytes, i):
    return (hash_bytes[i // 8] >> (7 - i % 8)) & 1

In [2]:
class Node:
    __slots__ = ("hash", "entries", "children")

    def __init__(self):
        self.hash = None
        self.entries = None
        self.children = None

In [3]:
def build_node(items, depth):
    node = Node()
    if len(items) <= MAX_LEAF or depth >= MAX_DEPTH:
        items = sorted(items)
        node.entries = dict(items)
        node.hash = h(b"".join(eh for _, eh in items))
        return node

    left, right = [], []
    for eid, eh in items:
        (left if bit(eh, depth) == 0 else right).append((eid, eh))
    node.children = (build_node(left, depth + 1), build_node(right, depth + 1))
    node.hash = h(node.children[0].hash + node.children[1].hash)
    return node


def build_trie(data: dict):
    items = [(eid, entry_hash(eid, xy)) for eid, xy in data.items()]
    return build_node(items, 0)


def split_leaf(node, depth):
    left, right = [], []
    for eid, eh in node.entries.items():
        (left if bit(eh, depth) == 0 else right).append((eid, eh))
    return build_node(left, depth + 1), build_node(right, depth + 1)


def diff(node_a, node_b, depth=0, max_leaf_seen=0):
    if node_a.hash == node_b.hash:
        return [], max_leaf_seen

    a_leaf, b_leaf = node_a.entries is not None, node_b.entries is not None

    if a_leaf and b_leaf:
        ids_a, ids_b = set(node_a.entries), set(node_b.entries)
        touched = max(len(ids_a), len(ids_b))
        result = [("A", eid) for eid in ids_a - ids_b] + [("B", eid) for eid in ids_b - ids_a]
        return result, max(max_leaf_seen, touched)

    a0, a1 = node_a.children if not a_leaf else split_leaf(node_a, depth)
    b0, b1 = node_b.children if not b_leaf else split_leaf(node_b, depth)
    d0, m0 = diff(a0, b0, depth + 1, max_leaf_seen)
    d1, m1 = diff(a1, b1, depth + 1, max(max_leaf_seen, m0))
    return d0 + d1, max(m0, m1)


def sync(robot_a, robot_b):
    start = time.perf_counter()

    trie_a = build_trie(robot_a.data)
    trie_b = build_trie(robot_b.data)
    diffs, largest_leaf = diff(trie_a, trie_b)

    for owner, eid in diffs:
        if owner == "A":
            robot_b.data[eid] = robot_a.data[eid]
        else:
            robot_a.data[eid] = robot_b.data[eid]

    elapsed = time.perf_counter() - start
    return elapsed, len(diffs), largest_leaf


class Robot:
    def __init__(self, robot_id, n_samples=1000, seed=None, skewed=False):
        rng = random.Random(seed)
        self.id = robot_id
        if not skewed:
            self.data = {
                f"{robot_id}-{i}": (rng.uniform(0, 100), rng.uniform(0, 100))
                for i in range(n_samples)
            }
        else:
            self.data = {
                f"{robot_id}-cluster-{i}": (rng.uniform(0, 1), rng.uniform(0, 1))
                for i in range(n_samples)
            }


if __name__ == "__main__":
    print("=== Uniform random keys, 5 robots x 1000 samples ===")
    robots = [Robot(f"R{i}", n_samples=1000, seed=i) for i in range(5)]
    total_time, total_exchanged, worst_leaf = 0.0, 0, 0
    for a, b in itertools.combinations(robots, 2):
        t, exch, leaf = sync(a, b)
        total_time += t
        total_exchanged += exch
        worst_leaf = max(worst_leaf, leaf)
    print(f"Total sync time: {total_time*1000:.2f} ms | entries exchanged: {total_exchanged} "
          f"| largest leaf ever compared: {worst_leaf}")

    print("\n=== Skewed/clustered keys (would blow up a fixed-bucket tree), 5 robots x 1000 samples ===")
    robots2 = [Robot(f"S{i}", n_samples=1000, seed=i, skewed=True) for i in range(5)]
    total_time, total_exchanged, worst_leaf = 0.0, 0, 0
    for a, b in itertools.combinations(robots2, 2):
        t, exch, leaf = sync(a, b)
        total_time += t
        total_exchanged += exch
        worst_leaf = max(worst_leaf, leaf)
    print(f"Total sync time: {total_time*1000:.2f} ms | entries exchanged: {total_exchanged} "
          f"| largest leaf ever compared: {worst_leaf}")

    print("\n=== Realistic case: robots already synced, only a few new points each ===")
    base = {f"shared-{i}": (random.uniform(0, 100), random.uniform(0, 100)) for i in range(1000)}
    robots3 = []
    for i in range(5):
        r = Robot(f"T{i}", n_samples=0, seed=i)
        r.data = dict(base)
        for j in range(5):
            r.data[f"T{i}-new-{j}"] = (random.uniform(0, 100), random.uniform(0, 100))
        robots3.append(r)
    total_time, total_exchanged, worst_leaf = 0.0, 0, 0
    for a, b in itertools.combinations(robots3, 2):
        t, exch, leaf = sync(a, b)
        total_time += t
        total_exchanged += exch
        worst_leaf = max(worst_leaf, leaf)
    print(f"Total sync time: {total_time*1000:.2f} ms | entries exchanged: {total_exchanged} "
          f"| largest leaf ever compared: {worst_leaf}")

=== Uniform random keys, 5 robots x 1000 samples ===
Total sync time: 400.77 ms | entries exchanged: 20000 | largest leaf ever compared: 4

=== Skewed/clustered keys (would blow up a fixed-bucket tree), 5 robots x 1000 samples ===
Total sync time: 520.98 ms | entries exchanged: 20000 | largest leaf ever compared: 4

=== Realistic case: robots already synced, only a few new points each ===
Total sync time: 85.44 ms | entries exchanged: 100 | largest leaf ever compared: 4
